# 02 — API: Weather Forecasts
This notebook fetches 5-day / 3-hour weather forecasts for each city from the OpenWeatherMap API and loads them into the `city_weather` table.

Weather is one of the strongest predictors of e-scooter demand — rain causes sharp drops in usage.

**Source:** [OpenWeatherMap Forecast API](https://openweathermap.org/forecast5)  
**Target table:** `city_weather`  
**Run frequency:** Daily (forecasts update every 3 hours)

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import requests
import getpass
from datetime import datetime, timezone

---
## 2. Credentials
Both the MySQL password and the OpenWeatherMap API key are entered at runtime.

In [ ]:
password        = getpass.getpass("MySQL password: ")
api_weather_key = getpass.getpass("OpenWeatherMap API key: ")

schema = "gans_db"
host   = "127.0.0.1"
user   = "root"
port   = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"

---
## 3. Extract — Fetch Weather Forecasts
City coordinates are read directly from the database, ensuring this notebook stays in sync with whatever cities have been loaded.

In [ ]:
# Read cities from DB — use their stored coordinates and IDs
cities_df = pd.read_sql("SELECT city_id, city_name, latitude, longitude FROM cities", con=connection_string)

FORECAST_URL = "https://api.openweathermap.org/data/2.5/forecast"
retrieved_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

weather_records = []

for _, city in cities_df.iterrows():
    params = {
        "lat":   city["latitude"],
        "lon":   city["longitude"],
        "appid": api_weather_key,
        "units": "metric"
    }
    response = requests.get(FORECAST_URL, params=params)

    if response.status_code != 200:
        print(f"[ERROR] {city['city_name']} — status {response.status_code}")
        continue

    for entry in response.json()["list"]:
        weather_records.append({
            "city_id":            city["city_id"],
            "forecast_time":      entry.get("dt_txt"),
            "retrieved_at":       retrieved_at,
            "description":        entry.get("weather", [{}])[0].get("main"),
            "temperature_C":      entry.get("main", {}).get("temp"),
            "precipitation_prob": entry.get("pop", 0),
            "rain_mm":            entry.get("rain", {}).get("3h", 0),
            "snow_mm":            entry.get("snow", {}).get("3h", 0),
            "wind_speed_ms":      entry.get("wind", {}).get("speed", 0),
            "visibility_m":       entry.get("visibility", 10000)
        })

weather_df = pd.DataFrame(weather_records)
print(f"Fetched {len(weather_df)} forecast entries across {len(cities_df)} cities.")
weather_df.head()

---
## 4. Transform — Clean & Type Check

In [ ]:
weather_df["forecast_time"] = pd.to_datetime(weather_df["forecast_time"])
weather_df["retrieved_at"]  = pd.to_datetime(weather_df["retrieved_at"])

# Clip precipitation probability to valid range
weather_df["precipitation_prob"] = weather_df["precipitation_prob"].clip(0, 1)

weather_df.dtypes

---
## 5. Load — Push to MySQL
`if_exists='append'` adds new forecasts without overwriting history.

Note: re-running this notebook on the same day will raise a duplicate key error on `(city_id, forecast_time)` — this is intentional and protects against double-inserting the same forecast.

In [ ]:
rows_inserted = weather_df.to_sql(
    "city_weather",
    if_exists="append",
    con=connection_string,
    index=False
)
print(f"Inserted {rows_inserted} rows into `city_weather`.")